In [ ]:
import numpy as np
import torch
import json

file_num = ""
filepath = f'/{file_num}.wav'

SILERO_MODEL, SILERO_UTILS = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', trust_repo=True)

DEFAULT_THRESHOLD = 0.25 # Lower for more sensitivity, higher for less sensitivity
MAX_VALUE = 32768.0

def get_speech_timestamps(model: torch.nn.Module, utils: tuple, audio_bytes: bytes, sample_rate: int=16000, threshold: float=DEFAULT_THRESHOLD) -> list[dict[str, float]]:
    """
    Uses Silero VAD to detect voiced segments in the audio bytes.
    """
    (get_speech_ts, _, _, _, _) = utils

    # Convert the byte stream into an array of floats
    audio_np = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / MAX_VALUE

    # The model expects a float32 tensor
    audio_tensor = torch.tensor(audio_np).unsqueeze(0)

    # Get timestamps where voice is detected
    speech_timestamps = get_speech_ts(audio_tensor, model, sampling_rate=sample_rate, threshold=threshold)

    # Update samples to timestamps
    for item in speech_timestamps:
        item['start'] = item['start'] / float(sample_rate)
        item['end'] = item['end'] / float(sample_rate)

    print(speech_timestamps)
    return speech_timestamps

def read_wav_file_to_bytes(file_path):
    with open(file_path, 'rb') as wav_file:
        audio_bytes = wav_file.read()
    return audio_bytes

# Testing model
# file_path = '../../../data/speech_and_instruments/VoiceGuitarHalfNHalf.wav'
# audio_bytes = read_wav_file_to_bytes(file_path)
# print(type(audio_bytes), len(audio_bytes)) 
# get_speech_timestamps(SILERO_MODEL, SILERO_UTILS, audio_bytes)

audio_bytes = read_wav_file_to_bytes(filepath)
print(type(audio_bytes), len(audio_bytes))

timestamps = get_speech_timestamps(SILERO_MODEL, SILERO_UTILS, audio_bytes)

print(json.dumps(timestamps))
